# Phase 1 — Explorer le référentiel, sans rien corriger (J1 matin)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from validation_structure import normaliser
from validation_structure import valider

#### Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?
  
Oui, les deux documents contiennent exactement la même chose une fois chargés.

In [10]:
import pandas as pd

data_csv = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)
data_xlsx = pd.read_excel("../data/numeros_tva.xlsx", dtype=str, keep_default_na=False)
for col in data_csv.columns:
    differences = data_csv[col] != data_xlsx[col]
    nb_diff = differences.sum()
    print(f"Colonne '{col}' : {nb_diff} différences")
    if nb_diff > 0:
        comparaison = pd.DataFrame({
            "id": data_csv.loc[differences, "id"],
            "CSV": data_csv.loc[differences, col],
            "Excel": data_xlsx.loc[differences, col]
        })
        print(comparaison.head(5))
        print("-" * 40)

Colonne 'id' : 0 différences
Colonne 'raison_sociale' : 0 différences
Colonne 'pays_declare' : 0 différences
Colonne 'numero_tva' : 0 différences
Colonne 'date_saisie' : 0 différences
Colonne 'source_saisie' : 0 différences


#### Combien de pays distincts ? Combien de formats différents pour un même pays ?

Il y a 15 pays distincts.

In [19]:
display(data_csv.describe())

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_normalise,est_valide,motif,rejets
count,10000,10000,10000,10000,10000,10000,10000,10000,10000,10000
unique,10000,3046,15,9304,640,5,9304,5,5,5
top,1,Global Distribution SpA,FR,,2024-11-25,reprise_erp,,"(True, ok)","(True, ok)",ok
freq,1,12,974,174,29,2071,174,6083,6083,6083


In [ ]:
data_csv["tva_normalise"] = data_csv["numero_tva"].apply(normaliser)
resultats = data_csv["tva_normalise"].apply(valider)
data_csv["est_valide"] = [res for res in resultats]
data_csv["motif"] = [res for res in resultats]
display("Répartition des motifs")
display(data_csv["motif"].value_counts())
display("Aperçu des 10 premières lignes")
display(data_csv[["numero_tva", "tva_normalise", "est_valide", "motif"]].head(10))


'Répartition des motifs'

motif
(True, ok)                   6083
(False, cle_invalide)        2605
(False, pays_non_couvert)    1090
(False, vide)                 174
(False, trop_court)            48
Name: count, dtype: int64

'Aperçu des 10 premières lignes'

,numero_tva,tva_normalise,est_valide,motif
0,DK73224645,DK73224645,"(False, cle_invalide)","(False, cle_invalide)"
1,PT787327239,PT787327239,"(True, ok)","(True, ok)"
2,LU71043234,LU71043234,"(True, ok)","(True, ok)"
3,SE751298146001,SE751298146001,"(True, ok)","(True, ok)"
4,NL505862176B88,NL505862176B88,"(False, cle_invalide)","(False, cle_invalide)"
5,DK61530843,DK61530843,"(True, ok)","(True, ok)"
6,NL505862176B88,NL505862176B88,"(False, cle_invalide)","(False, cle_invalide)"
7,SE314125231601,SE314125231601,"(True, ok)","(True, ok)"
8,DK48646344,DK48646344,"(True, ok)","(True, ok)"
9,BE1997422119,BE1997422119,"(True, ok)","(True, ok)"


,numero_tva,pays_declare


#### Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

il y a 2605 cle_invalides, soit 26.05%.

`print(data_csv[data_csv["rejets"] == "cle_invalide"].head(5))`


#### Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

il y a 1474 vides, soit 1.74%. 
Les valeurs "trop court" sont aussi des notations de vide : NA. Il y en a 48, soit .48%.

`print(data_csv[data_csv["numero_tva"].str.upper().str.startswith("NA")])`

je ne trouve pas les autres nan ou NAN ou autres. 

In [23]:
data_csv["rejets"] = data_csv["numero_tva"].apply(valider).str.get(1)
tous_les_rejets = ["ok", "vide", "trop_court", "pays_non_couvert", "cle_invalide"]
compte = data_csv["rejets"].value_counts().reindex(tous_les_rejets, fill_value=0)
pourcentage = (compte / len(data_csv) * 100).round(2)

display(compte, pourcentage)

rejets
ok                  6083
vide                 174
trop_court            48
pays_non_couvert    1090
cle_invalide        2605
Name: count, dtype: int64

rejets
ok                  60.83
vide                 1.74
trop_court           0.48
pays_non_couvert    10.90
cle_invalide        26.05
Name: count, dtype: float64

#### Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?

non, ça ne correspond pas pour 793 lignes

In [16]:
tva_norm = data_csv["numero_tva"].apply(normaliser)
prefixe = tva_norm.str[:2]
pays_crm = data_csv["pays_declare"].str.strip().str.upper()
pays_pas_prefixe = prefixe != pays_crm
display("Nombre d'incohérences :", pays_pas_prefixe.sum())
display(data_csv.loc[pays_pas_prefixe, ["id", "pays_declare", "numero_tva"]].head(15))

Nombre d'incohérences : 793
      id pays_declare    numero_tva
25    26           PT             -
80    81           IT              
91    92           FR              
119  120           FR   58560322811
134  135           NL  396904956B68
162  163           IT   38486825359
180  181           IT   33139885439
188  189           IT   88838354509
194  195           SE  107327005801
246  247           PL    5112134613
259  260           LU      75041556
261  262           BE    0572894866
263  264           IT   09184526433
267  268           IT              
269  270           LU      22312502


#### Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

Après avoir testé 2 méthodes, je n'observe pas de différence, aucun ne semble sortir du lot. Peut etre obligation de choisir une date dans un menu déroulant, forçant format ?

In [24]:
## 1er essai
# import re
# def est_format_iso(valeur: str) -> bool: 
#     pattern = r"^\d{4}-\d{2}-\d{2}$"
#     return bool(re.match(pattern, valeur))
# dates_ok = data_csv["date_saisie"].apply(est_format_iso)
# dates_pas_ok = ~dates_ok
# print(len(dates_pas_ok))
# print(data_csv[dates_pas_ok][["id", "date_saisie", "source_saisie"]].head(10))

## 2e essai : 
dates_pas_ok = ~data_csv["date_saisie"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
print(data_csv.loc[dates_pas_ok, "date_saisie"])

Series([], Name: date_saisie, dtype: str)


Ce code remplace définitivement le contenu de la colonne numero_tva par sa version nettoyée via la fonction normaliser, puis affiche les 5 premières lignes :

In [18]:
data_csv["numero_tva"] = data_csv["numero_tva"].map(normaliser)
print(data_csv["numero_tva"].head(5))

0        DK73224645
1       PT787327239
2        LU71043234
3    SE751298146001
4    NL505862176B88
Name: numero_tva, dtype: str


## En conclusion: 

Je choisis de travailler avec le document de données CSV : 
- plus rapide, 
- ~~plus facilement manipulable (fichier lourd pour excel)~~ ,
- des antécédants avec le titanic et les fleurs,
- besoin de practice python 

Réponse 11h : il y a moins de chances que quelqu'un ait modifié un CSV, plutôt qu'un excel

parquet sera le remplaçant de CSV à terme. Le parquet compresse les doublons en binaire et est plus léger